In [6]:
import pandas as pd

# 1. Définition des paramètres pour le Scénario de Base (Statut Quo)
# Les VRAIES données de 2025 issues de ton Notebook Q3
inputs_base_2025 = {
    "start_users": 1000,          # Base arbitraire de départ pour simuler la cohorte
    "monthly_ad_spend": 5000,     # Budget Demand constant pour comparer
    "cac": 21.0,                  # Moyenne entre Meta (18.8€) et Google (24.1€)
    "churn_rate": 0.32,           # ~32% de perte mensuelle (basé sur la chute M0 -> M1)
    "referral_rate": 0.06,        # Estimation prudente de la viralité (CAC 0€)
    "arpu": 10.5,                 # ~63€ sur 6 mois = 10.5€ par mois en moyenne
    "margin": 1.0                 # Déjà inclus dans ton calcul de kozy_revenue
}

# 2. Fonction de modélisation du Growth Loop
def simulate_growth(scenario_name, params, months=12):
    results = []
    current_users = params["start_users"]
    
    for m in range(months + 1):
        if m == 0:
            results.append({
                "Month": m, 
                "Start users": None, "Churned": None, "Retained": None, 
                "Paid new users": None, "Referrals": None, 
                "End users": current_users, 
                "Revenue (€)": 0
            })
            continue
            
        # Application de la formule du simulateur IMT : End users = Start users × (1 − churn) × (1 + referral rate) + Ad spend / CAC[cite: 6]
        churned = current_users * params["churn_rate"]
        retained = current_users - churned
        paid_new = params["monthly_ad_spend"] / params["cac"]
        referrals = retained * params["referral_rate"]
        
        current_users = retained + paid_new + referrals
        revenue = current_users * params["arpu"] * params["margin"]
        
        results.append({
            "Month": m,
            "Start users": retained + churned,
            "Churned": churned,
            "Retained": retained,
            "Paid new users": paid_new,
            "Referrals": referrals,
            "End users": current_users,
            "Revenue (€)": revenue
        })
        
    df_results = pd.DataFrame(results)
    
    # Ajout des métriques de la ligne "M12" du simulateur[cite: 7]
    mrr_m12 = df_results.iloc[-1]["Revenue (€)"]
    print(f"--- {scenario_name} ---")
    print(f"Utilisateurs au Mois 12 : {df_results.iloc[-1]['End users']:.0f}")
    print(f"MRR au Mois 12 : {mrr_m12:,.0f} €\n")
    
    return df_results

# 3. Exécution du Backtest / Scénario de Base
df_base = simulate_growth("Scénario 1: Base (Statut Quo)", inputs_base_2025, months=12)

# Affichage des 6 premiers mois pour vérification (similaire à la slide EXAMPLE)[cite: 10]
display(df_base.head(7).round(1))

--- Scénario 1: Base (Statut Quo) ---
Utilisateurs au Mois 12 : 856
MRR au Mois 12 : 8,985 €



,Month,Start users,Churned,Retained,Paid new users,Referrals,End users,Revenue (€)
0,0,NaN,NaN,NaN,NaN,NaN,1000.0,0.0
1,1,1000.0,320.0,680.0,238.1,40.8,958.9,10068.4
2,2,958.9,306.8,652.0,238.1,39.1,929.3,9757.3
3,3,929.3,297.4,631.9,238.1,37.9,907.9,9533.1
4,4,907.9,290.5,617.4,238.1,37.0,892.5,9371.4
5,5,892.5,285.6,606.9,238.1,36.4,881.4,9254.9
6,6,881.4,282.1,599.4,238.1,36.0,873.4,9171.0


We should analyse this result and compare it to revenue and other metrics whithout model. (meaning with revenue at month 12)?
Plus analyse results (same goes forscenarios, graphs too)

In [7]:
import pandas as pd

# 1. Parameter Definitions (Derived from 2025 Historical Data)
base_params = {
    "start_users": 1000,          # Initial user base for the cohort model
    "monthly_ad_spend": 5000,     # Constant monthly budget for demand generation
    "cac": 21.0,                  # Average CAC (Meta Ads 18.8€, Google Ads 24.1€)
    "churn_rate": 0.32,           # Observed early churn (drop from ~76% in M0 to ~52% in M1)
    "referral_rate": 0.06,        # Organic/Referral growth factor
    "arpu": 10.5,                 # ~63€ LTV over 6 months -> ~10.5€ per month
    "margin": 1.0                 # Margin already factored into ARPU
}

# Optimistic Scenario (Rebalancing budget to Supply in Lyon)
opt_params = base_params.copy()
opt_params["churn_rate"] = 0.15   # Fulfillment reaches 90%, churn drops significantly
opt_params["referral_rate"] = 0.12 # Happy customers refer more

# Pessimistic Scenario (Ignoring Supply crisis)
pess_params = base_params.copy()
pess_params["cac"] = 40.0         # CAC doubles as conversion plummets
pess_params["churn_rate"] = 0.45  # Nearly half of customers leave monthly

scenarios = {
    "Base Scenario (Status Quo)": base_params,
    "Optimistic (Rebalanced Supply)": opt_params,
    "Pessimistic (Liquidity Crisis)": pess_params
}

# 2. Execute Scenarios
for name, params in scenarios.items():
    simulate_growth(name, params)

--- Base Scenario (Status Quo) ---
Utilisateurs au Mois 12 : 856
MRR au Mois 12 : 8,985 €

--- Optimistic (Rebalanced Supply) ---
Utilisateurs au Mois 12 : 2766
MRR au Mois 12 : 29,039 €

--- Pessimistic (Liquidity Crisis) ---
Utilisateurs au Mois 12 : 301
MRR au Mois 12 : 3,159 €

